In [78]:
import jax
import netket as nk
from copy import deepcopy

import numpy as np
import jax.numpy as jnp

# from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule
from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule 


In [79]:
mol, mo_coeff, mf = PCMolecule.molecule(cid=62714)#62714
molecule = PCMolecule(mol=mol, mo_coeff=mo_coeff)

H = molecule.hamiltonian.to_jax_operator()
hi = molecule.hilbert_space

using 2d
Hartree-Fock energy: -7.767362135748573
E(CCSD) = -7.784454825913955  E_corr = -0.01709269016538233
CCSD energy: -7.784454825913955


/Users/lucagravina/venvs/neuralimportancesampling/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [80]:
all_states = hi.all_states()
print("all_states_size =", all_states.shape)

_operator_data = H._operator_data
print("_operator_data keys =", _operator_data.keys())

all_states_size = (225, 12)
_operator_data keys = dict_keys(['diag', 'offdiag'])


In [4]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            k: v for k, v in inner_dict.items() 
            if filter_func(k)
        }
    return result


_operator_data_filtered = filter_keys(_operator_data, lambda k: k == 4)
_operator_data_filtered['diag'] = {}

In [14]:
import jax.ops
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc
from netket.experimental.operator._particle_number_conserving_fermionic._matrix_elements import _get_mel_offdiag

x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

print("xp shape =", xp.shape)
print("mels shape =", mels.shape)

xp shape = (35, 12)
mels shape = (35,)


In [15]:
k = 4

index_array, create_array, weight_array = _operator_data_filtered['offdiag'][4]

mels_off_diag = _get_mel_offdiag(
    x,
    xp,
    index_array,
    create_array,
    weight_array,
)

np.testing.assert_allclose(mels, mels_off_diag)

In [ ]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels_truncated import get_conn_padded_pnc_truncated

x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])
xp, mels = get_conn_padded_pnc(_operator_data, x, hi.n_fermions)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

y = xp.copy()

xp_truncated, mels_truncated = get_conn_padded_pnc_truncated(
    _operator_data,
    x,
    y,
    hi.n_fermions,
)
xp_truncated, inverse_indices = jnp.unique(xp_truncated, axis=0, return_inverse=True)
mels_truncated = jax.ops.segment_sum(mels_truncated, inverse_indices, num_segments=len(xp_truncated))

np.testing.assert_allclose(xp_truncated, xp)
np.testing.assert_allclose(mels_truncated, mels)


In [102]:
x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

all_connected, _ = get_conn_padded_pnc(_operator_data, x, hi.n_fermions)
y_reduced = all_connected[0:5]

_operator_data_reduced = deepcopy(_operator_data)
_operator_data_reduced['offdiag'].pop(4)

xp_reduced, mels_reduced = get_conn_padded_pnc(_operator_data_reduced, x, hi.n_fermions)

_operator_data_twobody_offdiag = deepcopy(_operator_data)
_operator_data_twobody_offdiag['diag'] = {}
_operator_data_twobody_offdiag['offdiag'].pop(2)

xp_twobody_offdiag, mels_twobody_offdiag = get_conn_padded_pnc(_operator_data_twobody_offdiag, x, hi.n_fermions)
mask = (xp_twobody_offdiag[:, None] == y_reduced[None, :]).all(axis=-1).any(axis=-1)
xp_twobody_offdiag_truncated = xp_twobody_offdiag[mask]
mels_twobody_offdiag_truncated = mels_twobody_offdiag[mask]

xp = jnp.concatenate([xp_reduced, xp_twobody_offdiag_truncated], axis=-2)
mels = jnp.concatenate([mels_reduced, mels_twobody_offdiag_truncated], axis=-1)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

In [110]:
xp_truncated, mels_truncated = get_conn_padded_pnc_truncated(
    _operator_data,
    x,
    y_reduced,
    hi.n_fermions,
)
xp_truncated, inverse_indices = jnp.unique(xp_truncated, axis=0, return_inverse=True)
mels_truncated = jax.ops.segment_sum(mels_truncated, inverse_indices, num_segments=len(xp_truncated))

np.testing.assert_allclose(xp, xp_truncated)
np.testing.assert_allclose(mels, mels_truncated)